# Premier League fixture preparation
Combine the two season CSVs, compute pre-match form, verify it, and export separate inputs, targets and benchmark odds.

Run all cells in Python 3.9+. Put the named CSVs next to the notebook or in its `upload/` folder. No third-party packages are required. Add future season files in the configuration cell.

**Availability rule.** The CSVs provide kickoff times but no final-whistle or publication timestamps. This pilot uses only results from earlier calendar dates in the source time convention. This deliberately skips same-day results. Form includes home and away matches and carries over between seasons for teams with recorded history. A team returning to the league may have older recorded matches. Rows with fewer than three prior matches on either side stay in the export with a readiness flag and blank totals on the incomplete side.


In [1]:
from collections import defaultdict
import csv
from datetime import datetime
from pathlib import Path

SEASON_FILES = {
    "2023/24": "E 2023 to 2024.csv",
    "2024/25": "E 2024 to 2025.csv",
}
WINDOW = 3
OUTPUT = Path("fixture_outputs")
ODDS_GROUPS = {
    "average_1x2": ("AvgCH", "AvgCD", "AvgCA"),
    "average_total_25": ("AvgC>2.5", "AvgC<2.5"),
    "bet365_1x2": ("B365CH", "B365CD", "B365CA"),
    "bet365_total_25": ("B365C>2.5", "B365C<2.5"),
}
ODDS_COLUMNS = [column for group in ODDS_GROUPS.values() for column in group]

def load_fixtures(season_files):
    fixtures = []
    seen = set()
    for season, filename in season_files.items():
        path = Path(filename)
        if not path.is_file():
            path = Path("upload") / filename
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for line, row in enumerate(csv.DictReader(stream), start=2):
                try:
                    kickoff = datetime.strptime(
                        f"{row['Date']} {row['Time']}", "%d/%m/%Y %H:%M"
                    )
                    home, away = row["HomeTeam"].strip(), row["AwayTeam"].strip()
                    hg, ag = int(row["FTHG"]), int(row["FTAG"])
                    if not home or not away or home == away or min(hg, ag) < 0:
                        raise ValueError("Invalid teams or score")
                    if row["FTR"] != ("H" if hg > ag else "A" if hg < ag else "D"):
                        raise ValueError("Result disagrees with full-time score")
                except (KeyError, ValueError, TypeError, AttributeError) as error:
                    raise ValueError(f"{path}:{line}: {error}") from error
                fixture_id = f"{season}|{home}|{away}"
                if fixture_id in seen:
                    raise ValueError(f"Duplicate fixture: {fixture_id}")
                seen.add(fixture_id)
                fixtures.append(dict(
                    fixture_id=fixture_id, season=season, kickoff=kickoff,
                    home_team=home, away_team=away, home_goals=hg, away_goals=ag,
                    **{column: row.get(column, "") for column in ODDS_COLUMNS}
                ))
    if not fixtures:
        raise ValueError("No fixtures found")
    return sorted(fixtures, key=lambda row: (row["kickoff"], row["fixture_id"]))

fixtures = load_fixtures(SEASON_FILES)
for season in SEASON_FILES:
    print(season, sum(row["season"] == season for row in fixtures), "fixtures")


2023/24 380 fixtures
2024/25 380 fixtures


## Build reusable form features

The function accepts completed history and a separate list of fixtures to forecast. Upcoming fixtures need only the fixture ID, season, kickoff and team names. In a historical replay, pass the same fixture list in both arguments. The date check excludes the target fixture and all later results.


In [2]:
def build_form_features(completed, targets, window=3):
    if not isinstance(window, int) or window < 1:
        raise ValueError("The form window must be a positive integer")

    history = defaultdict(list)
    for match in sorted(completed, key=lambda row: row["kickoff"]):
        hg, ag = match["home_goals"], match["away_goals"]
        for side, scored, conceded in (("home", hg, ag), ("away", ag, hg)):
            points = 3 if scored > conceded else 1 if scored == conceded else 0
            history[match[f"{side}_team"]].append(
                (match["kickoff"].date(), points, scored, conceded)
            )

    rows = []
    for match in sorted(targets, key=lambda row: (row["kickoff"], row["fixture_id"])):
        features = {key: match[key] for key in (
            "fixture_id", "season", "kickoff", "home_team", "away_team"
        )}
        for side in ("home", "away"):
            prior = [
                entry for entry in history[match[f"{side}_team"]]
                if entry[0] < match["kickoff"].date()
            ][-window:]
            features[f"{side}_prior_count_capped"] = len(prior)
            for index, name in enumerate(("points", "goals_scored", "goals_conceded"), 1):
                features[f"{side}_{name}_last_{window}"] = (
                    sum(entry[index] for entry in prior) if len(prior) == window else None
                )
        features["form_ready"] = all(
            features[f"{side}_prior_count_capped"] == window for side in ("home", "away")
        )
        rows.append(features)
    return rows

features = build_form_features(fixtures, fixtures, WINDOW)
print(sum(row["form_ready"] for row in features), "fixtures with full form for both teams")
print(features[0])


720 fixtures with full form for both teams
{'fixture_id': '2023/24|Burnley|Man City', 'season': '2023/24', 'kickoff': datetime.datetime(2023, 8, 11, 20, 0), 'home_team': 'Burnley', 'away_team': 'Man City', 'home_prior_count_capped': 0, 'home_points_last_3': None, 'home_goals_scored_last_3': None, 'home_goals_conceded_last_3': None, 'away_prior_count_capped': 0, 'away_points_last_3': None, 'away_goals_scored_last_3': None, 'away_goals_conceded_last_3': None, 'form_ready': False}


## Create outcomes and keep benchmark odds separate

Join exports on `fixture_id`. The modelling inputs are the lagged totals. Identifiers, kickoff, season, and readiness fields are included for tracking and chronological splits. The benchmark file retains raw closing odds as supplied. Exact capture timestamps are unavailable in these files. Score labels and odds are in separate exports.


In [3]:
def make_targets(match):
    hg, ag = match["home_goals"], match["away_goals"]
    return {
        "fixture_id": match["fixture_id"],
        "home_goals": hg,
        "away_goals": ag,
        "target_1x2": "H" if hg > ag else "A" if hg < ag else "D",
        "target_over_25": int(hg + ag >= 3),
        "target_btts": int(hg > 0 and ag > 0),
    }

targets = [make_targets(match) for match in fixtures]
odds = [
    {key: match[key] for key in ["fixture_id", *ODDS_COLUMNS]}
    for match in fixtures
]


## Verify on real fixtures

For Arsenal vs Man United on 3 September 2023, Arsenal's prior scores from its perspective were 2–1 vs Nott'm Forest, 1–0 vs Crystal Palace and 2–2 vs Fulham. That gives **7 points, 5 scored, 3 conceded**.

For Man City vs Fulham on 2 September 2023, City's prior scores were 3–0 vs Burnley, 1–0 vs Newcastle and 2–1 vs Sheffield United. That gives **9 points, 6 scored, 1 conceded**. The checks below also perturb a result and verify its features are unchanged while a later fixture's features respond.


In [4]:
assert [r["fixture_id"] for r in features] == [r["fixture_id"] for r in targets]
assert [r["fixture_id"] for r in features] == [r["fixture_id"] for r in odds]
by_id = {row["fixture_id"]: row for row in features}
if WINDOW == 3 and "2023/24" in SEASON_FILES:
    for fixture_id, expected in {
        "2023/24|Arsenal|Man United": (7, 5, 3),
        "2023/24|Man City|Fulham": (9, 6, 1),
    }.items():
        actual = tuple(by_id[fixture_id][f"home_{name}_last_3"] for name in (
            "points", "goals_scored", "goals_conceded"
        ))
        assert actual == expected, (fixture_id, actual, expected)
        print(f"Manual check passed: {fixture_id} → {actual}")

changed_match = fixtures[0]
changed_history = [dict(changed_match, home_goals=changed_match["home_goals"] + 1), *fixtures[1:]]
changed_features = build_form_features(changed_history, fixtures, WINDOW)
same_date = changed_match["kickoff"].date()
assert all(
    original == changed
    for original, changed in zip(features, changed_features)
    if original["kickoff"].date() == same_date
)
if WINDOW == 3 and "2023/24" in SEASON_FILES:
    assert any(original != changed for original, changed in zip(features, changed_features))
print("Result-change check passed: own and same-day features are unchanged.")


Manual check passed: 2023/24|Arsenal|Man United → (7, 5, 3)
Manual check passed: 2023/24|Man City|Fulham → (9, 6, 1)
Result-change check passed: own and same-day features are unchanged.


In [5]:
OUTPUT.mkdir(exist_ok=True)
for filename, rows in (
    ("prematch_features.csv", features),
    ("match_targets.csv", targets),
    ("benchmark_odds.csv", odds),
):
    path = OUTPUT / filename
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"{path}: {len(rows)} rows")


fixture_outputs/prematch_features.csv: 760 rows
fixture_outputs/match_targets.csv: 760 rows
fixture_outputs/benchmark_odds.csv: 760 rows
